## Imports

In [8]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

In [9]:
from funcoes_escoragem import *

## Diretório

In [10]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [11]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [12]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) <= CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,96793066153,4171408,2026-06-21,2026-06-21,1,NI,4658.000000000,BLEND_REGRESSAO_2026,4171408,"{""pessoas"":[{""nome"":""GLAUCE MIRIAM MAIA DOS SA...",...,31,1778785248777,"{""valor_aluguel"":""1500"",""imobiliaria_id"":31865...",2026-06-21 18:00:31+00:00,1782064830973824661,96793066153,A,2026-06-21 18:07:50.314000+00:00,2026-06-21 14:12:51+00:00,APROVAR
1,1283714981,4190209,2026-06-22,2026-06-24,1,NI,2534.500000000,BLEND3_3,4190209,"{""pessoas"":[{""nome"":""MARIANNE GUADALUPE BUSS"",...",...,34,1778785248777,"{""valor_aluguel"":""1650"",""imobiliaria_id"":49424...",2026-06-25 08:00:36+00:00,1782374432338005807,01283714981,C,2026-06-25 08:00:46.151000+00:00,2026-06-24 16:09:12+00:00,DERIVAR
2,14123510906,4200655,2026-06-15,2026-06-16,1,NI,1164.500000000,BLEND3_3,4200655,"{""pessoas"":[{""nome"":""EMILY MAYARA PATRICIO BAR...",...,33,1778785248777,"{""valor_aluguel"":""1150"",""imobiliaria_id"":49038...",2026-06-16 18:00:37+00:00,1781632831598836383,14123510906,E,2026-06-16 18:03:52.430000+00:00,2026-06-16 09:32:02+00:00,APROVAR
3,515453900,4221089,2026-06-24,2026-06-24,1,B,4863.500000000,BLEND3_3,4221089,"{""pessoas"":[{""nome"":""JULIANA TAVARES EVANGELIS...",...,34,1778785248777,"{""valor_aluguel"":""2000"",""imobiliaria_id"":993,""...",2026-06-24 18:00:46+00:00,1782324043160590541,00515453900,C,2026-06-24 18:25:11.602000+00:00,2026-06-24 13:42:41+00:00,APROVAR
4,11591287740,4223841,2026-06-16,2026-06-16,1,NI,2055.000000000,BLEND3_3,4223841,"{""pessoas"":[{""nome"":""LAUDIMEIRE LOPES CALDEIRA...",...,32,1778785248777,"{""valor_aluguel"":""3600"",""imobiliaria_id"":18557...",2026-06-16 18:00:37+00:00,1781632832875305338,11591287740,B,2026-06-16 18:03:52.898000+00:00,2026-06-16 10:37:05+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111823,16632889689,4360650,2026-07-11,2026-07-11,1,NI,2123.500000000,BLEND3_3,4360650,"{""pessoas"":[{""nome"":""YAGO ADRIANO BARBOSA FELI...",...,32,1778785248777,"{""valor_aluguel"":900,""matchmaking_on"":false,""p...",2026-07-11 18:00:41+00:00,1783792839636203264,16632889689,B,2026-07-11 18:10:38.687000+00:00,2026-07-11 11:05:45+00:00,APROVAR
111824,12048381430,4360656,2026-07-11,2026-07-11,1,,0E-9,BLEND_4,4360656,"{""pessoas"":[{""nome"":"""",""documento"":""1204838143...",...,33,1778785248777,"{""principalDocument"":""12048381430"",""imobiliari...",2026-07-11 18:00:41+00:00,1783792839639888100,12048381430,E,2026-07-11 18:10:38.689000+00:00,2026-07-11 11:06:47+00:00,REPROVAR
111825,8247817918,4361121,2026-07-11,2026-07-11,1,E,6439.000000000,BLEND3_3,4361121,"{""pessoas"":[{""nome"":""JEIEL MAGALHAES RIBEIRO P...",...,33,1778785248777,"{""valor_aluguel"":1200,""matchmaking_on"":false,""...",2026-07-12 08:00:38+00:00,1783843237075837325,08247817918,E,2026-07-12 08:00:55.455000+00:00,2026-07-11 16:49:28+00:00,REPROVAR
111826,22324380234,4361336,2026-07-12,2026-07-12,1,NI,2329.000000000,BLEND3_3,4361336,"{""pessoas"":[{""nome"":""LIANE NAZARE CORREA FREIT...",...,33,1778785248777,"{""valor_aluguel"":1600,""matchmaking_on"":false,""...",2026-07-12 18:00:25+00:00,1783879224145719167,22324380234,E,2026-07-12 18:02:55.972000+00:00,2026-07-12 14:42:34+00:00,REPROVAR


In [13]:
credpago_df[credpago_df["contract_id"] == 4317077]

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result


In [14]:
# credpago_clean[credpago_clean["contract_id"] == 4317077][["contract_id", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

In [15]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [16]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                95532
BLEND_REGRESSAO_2026     6502
HVA3                     4401
BLEND_4                  3152
BVS_CUSTOM               1469
HVA4                      653
BVS_CUSTOM_V2              94
                           15
HFT1                       10
Name: count, dtype: int64

## Multiproponente

In [17]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    107504
2      3976
3       315
4        30
8         1
6         1
5         1
Name: count, dtype: Int64

In [18]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.961333
2    0.035555
3    0.002817
4    0.000268
8    0.000009
6    0.000009
5    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [19]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [20]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [21]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [22]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [23]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [24]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [25]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
830,0.000000,0.017544
855,0.000000,0.151584
856,0.000000,0.053846
862,0.000000,0.000000
863,0.000000,0.142132
...,...,...
110439,0.142857,0.094241
110440,0.235465,0.147808
110461,0.000000,0.087567
111555,0.000000,0.151584


## Escoragem Blend4

In [26]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [27]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [28]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [29]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] < 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_hadOverdueInvoiceInPreviousContracts,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,96793066153,4171408,2026-06-21,2026-06-21,1,NI,BLEND_REGRESSAO_2026,5979166,31,96793066153,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,4658.0
1,1283714981,4190209,2026-06-22,2026-06-24,1,NI,BLEND3_3,5999332,34,01283714981,...,NaN,NaN,NaN,4.0,20469.0,"CRED CARTAO,OUTRAS OPER",BLEND4,NaN,NaN,2534.5
2,14123510906,4200655,2026-06-15,2026-06-16,1,NI,BLEND3_3,5952156,33,14123510906,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1164.5
3,515453900,4221089,2026-06-24,2026-06-24,1,B,BLEND3_3,5997554,34,00515453900,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,4863.5
4,11591287740,4223841,2026-06-16,2026-06-16,1,NI,BLEND3_3,5952970,32,11591287740,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,2055.0


In [30]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111513
E_BVS        315
dtype: int64

In [31]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [32]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [33]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [34]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [35]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,96793066153,4171408,2026-06-21,2026-06-21,1,NI,BLEND_REGRESSAO_2026,5979166,31,96793066153,...,0.0,0.65,0.0,-0.238380,0.000000,0.000000,0.484682,515.0,A,E
1,1283714981,4190209,2026-06-22,2026-06-24,1,NI,BLEND3_3,5999332,34,01283714981,...,0.0,-0.25,5.0,0.488323,0.000000,-0.828362,0.530909,469.0,B,A
2,14123510906,4200655,2026-06-15,2026-06-16,1,NI,BLEND3_3,5952156,33,14123510906,...,0.0,-0.80,0.0,1.231688,0.000000,0.128657,0.544795,455.0,B,D
3,515453900,4221089,2026-06-24,2026-06-24,1,B,BLEND3_3,5997554,34,00515453900,...,0.0,0.65,0.0,-0.041347,-0.027615,-0.436110,0.528940,471.0,B,B
4,11591287740,4223841,2026-06-16,2026-06-16,1,NI,BLEND3_3,5952970,32,11591287740,...,0.0,0.30,0.0,2.919896,0.000000,0.000000,0.582527,417.0,C,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111823,16632889689,4360650,2026-07-11,2026-07-11,1,NI,BLEND3_3,6081704,32,16632889689,...,0.0,-0.55,0.0,-0.013510,0.000000,-1.305854,0.465513,534.0,A,A
111824,12048381430,4360656,2026-07-11,2026-07-11,1,,BLEND_4,6081712,33,12048381430,...,0.0,0.00,0.0,0.000000,0.000000,0.000000,0.620596,379.0,D,None
111825,8247817918,4361121,2026-07-11,2026-07-11,1,E,BLEND3_3,6082288,33,08247817918,...,0.0,-0.30,32.0,-0.538044,-0.005731,-0.397626,0.737757,262.0,E,E
111826,22324380234,4361336,2026-07-12,2026-07-12,1,NI,BLEND3_3,6082552,33,22324380234,...,0.0,1.35,1.0,0.567787,0.000000,1.867673,0.595004,405.0,D,E


In [36]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           15
BLEND3_3                95532
BLEND_4                  3152
BLEND_REGRESSAO_2026     6502
BVS_CUSTOM               1469
BVS_CUSTOM_V2              94
HFT1                       10
HVA3                     4401
HVA4                      653
dtype: int64

In [37]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [38]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111513
E_BVS        315
dtype: int64

In [39]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           15
BLEND3_3                95532
BLEND_4                  3152
BLEND_REGRESSAO_2026     6502
BVS_CUSTOM               1469
BVS_CUSTOM_V2              94
HFT1                       10
HVA3                     4401
HVA4                      653
dtype: int64

In [40]:
cp_final_df.groupby("model", dropna=False).size()

model
                           15
BLEND3_3                95532
BLEND_4                  3152
BLEND_REGRESSAO_2026     6502
BVS_CUSTOM               1469
BVS_CUSTOM_V2              94
HFT1                       10
HVA3                     4401
HVA4                      653
dtype: int64

In [41]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111513
E_BVS        315
dtype: int64

In [42]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)

## Investigação Sexta-Feira (03/07)

In [43]:
cp_final_df[
    (pd.to_datetime(cp_final_df["requested_at"]).dt.normalize() == "2026-07-03")
    & (cp_final_df["message_decisao"] == "BLEND_4")
    & (cp_final_df["pred_blend4_1_to_score"] != cp_final_df["message_blendRegressaoPredict"])
][id_vars + vars_blend4 + ["pred_blend4_1_to_score", "message_blendRegressaoPredict"]]

,contract_id,requested_at,CPF_CNPJ,message_decisao,message_blendRegressaoPredict,score_proposto__bvs,SERASA_CHSV5,age,property_type,qtde_restricoes__consulta_realizada,rental_value,income,agency_pc4_mais_100_contratos__pc_categorias,city_pc4_mais_100_contratos__pc_categorias,flag_tem__contratos_anteriores,flag_teve_boleto_atrasado__contratos_anteriores,pred_blend4_1_to_score,message_blendRegressaoPredict
830,4329626,2026-07-03,11426887124,BLEND_4,516.0,597.0,293.0,28.0,1,0,2542.0,1849.5,0.0,0.017544,0.0,0.0,483.0,516.0
855,4331837,2026-07-03,4094758151,BLEND_4,197.0,440.0,269.0,22.0,1,20,1050.0,1575.5,0.0,0.151584,0.0,0.0,177.0,197.0
856,4331891,2026-07-03,70007400608,BLEND_4,426.0,513.0,293.0,25.0,1,5,1200.0,2260.5,0.0,0.053846,0.0,0.0,393.0,426.0
862,4332210,2026-07-03,5426127042,BLEND_4,NaN,297.0,468.0,34.0,1,0,1400.0,0.0,0.0,0.000000,0.0,0.0,402.0,NaN
863,4332240,2026-07-03,3335741099,BLEND_4,152.0,335.0,97.0,32.0,1,15,1650.0,15892.0,0.0,0.142132,0.0,0.0,136.0,152.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108197,4331835,2026-07-03,50672991934,BLEND_4,521.0,488.0,579.0,69.0,1,0,2300.0,4521.0,0.0,0.107708,0.0,0.0,487.0,521.0
109293,4329890,2026-07-03,46224044864,BLEND_4,587.0,578.0,571.0,26.0,1,0,1300.0,1438.5,0.0,0.000000,0.0,0.0,628.0,587.0
109305,4331786,2026-07-03,27805930910,BLEND_4,526.0,524.0,572.0,70.0,1,0,2300.0,2055.0,0.0,0.107708,1.0,0.0,492.0,526.0
110461,4332395,2026-07-03,2565018070,BLEND_4,824.0,798.0,830.0,34.0,1,0,900.0,2466.0,0.0,0.087567,0.0,0.0,804.0,824.0
